# T08 - 汇率因子检验

## 项目概述

汇率因子检验项目专注于研究汇率变化对债券市场的影响。通过分析人民币汇率与债券收益率、利差等指标的相关性，识别汇率因子对债券投资的影响机制，为投资决策提供参考。

### 功能描述
1. **数据采集**：获取人民币兑美元汇率、资金利率、股指、债券收益率等金融数据
2. **因子分析**：计算汇率与各因子的静态相关性和滚动相关性
3. **高相关性识别**：识别汇率与各因子的高相关性时期
4. **可视化报告**：生成交互式HTML报告，包含相关性分析图表

### 数据源
- MySQL数据库（edb.edbdata、stock.indexcloseprice、bond.marketinfo_curve）
- 汇率数据：人民币兑美元即期汇率
- 资金利率：DR007、GC007
- 股指：沪深300、万得全A、中小板指、恒生指数
- 债券：10年国债、1年国债、5年城投债

### 输出结果
- 处理后的数据文件（processed_data.parquet）
- 相关性分析HTML报告（correlation_analysis.html）

---

## 1. 环境配置

导入依赖包并设置环境参数

In [ ]:
import os
import sys
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sqlalchemy
from sqlalchemy import pool
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyarrow

# 添加项目路径
sys.path.append(os.path.dirname(os.path.abspath('__file__')))

# 忽略警告
warnings.filterwarnings('ignore')

# 设置pandas显示选项
pd.set_option('display.unicode.ambiguous_as_wide', True)
pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', None)

print("环境配置完成！")
print(f"Python版本: {sys.version}")
print(f"Pandas版本: {pd.__version__}")
print(f"NumPy版本: {np.__version__}")

---

## 2. 数据获取

连接数据库并获取汇率及相关金融数据

In [ ]:
# 导入配置
from config import DB_CONFIG, ANALYSIS_CONFIG

# 创建数据库连接
def create_db_connection():
    """创建数据库连接"""
    connection_string = 'mysql+pymysql://%s:%s@%s:%s/%s' % (
        DB_CONFIG['user'],
        DB_CONFIG['password'],
        DB_CONFIG['host'],
        DB_CONFIG['port'],
        DB_CONFIG['database']
    )
    engine = sqlalchemy.create_engine(
        connection_string, 
        poolclass=sqlalchemy.pool.NullPool
    )
    return engine.connect()

# 测试连接
try:
    db_conn = create_db_connection()
    print("数据库连接成功！")
except Exception as e:
    print(f"数据库连接失败: {e}")
    print("将使用本地缓存数据...")
    db_conn = None

In [ ]:
# 数据获取函数
def get_edb_data(cursor, trade_code, bd, ed):
    """从edb.edbdata表获取数据"""
    sql = f"""
    SELECT DT as date, close as value 
    FROM edb.edbdata
    WHERE trade_code='{trade_code}'
    AND DT>='{bd}' AND DT <= '{ed}'
    ORDER BY DT
    """
    df = pd.read_sql(sql, cursor)
    df['date'] = pd.to_datetime(df['date'])
    return df

def get_stock_index_data(cursor, trade_code, bd, ed):
    """从stock.indexcloseprice表获取数据"""
    sql = f"""
    SELECT DT as date, close as value 
    FROM stock.indexcloseprice
    WHERE trade_code='{trade_code}'
    AND DT>='{bd}' AND DT <= '{ed}'
    ORDER BY DT
    """
    df = pd.read_sql(sql, cursor)
    df['date'] = pd.to_datetime(df['date'])
    return df

def get_bond_curve_data(cursor, trade_code, bd, ed):
    """从bond.marketinfo_curve表获取数据"""
    sql = f"""
    SELECT DT as date, close as value 
    FROM bond.marketinfo_curve
    WHERE trade_code='{trade_code}'
    AND DT>='{bd}' AND DT <= '{ed}'
    ORDER BY DT
    """
    df = pd.read_sql(sql, cursor)
    df['date'] = pd.to_datetime(df['date'])
    return df

print("数据获取函数定义完成！")

In [ ]:
# 获取所有数据
def get_all_data(cursor, bd, ed):
    """获取所有数据并进行预处理"""
    # 获取因变量：人民币兑美元即期汇率
    usd_cny = get_edb_data(cursor, 'M0067855', bd, ed)
    usd_cny = usd_cny.rename(columns={'value': 'USD_CNY'})
    
    # 获取目标变量
    data_dict = {
        'DR007': get_edb_data(cursor, 'M1006337', bd, ed),
        'GC007': get_edb_data(cursor, 'M1004515', bd, ed),
        'CSI300': get_edb_data(cursor, 'M0020209', bd, ed),
        'WIND_ALL_A': get_stock_index_data(cursor, '881001.WI', bd, ed),
        'SME_INDEX': get_stock_index_data(cursor, '399101.SZ', bd, ed),
        'HANG_SENG': get_stock_index_data(cursor, 'HSCI.HI', bd, ed),
        'BOND_10Y': get_bond_curve_data(cursor, 'L001619604', bd, ed),
        'BOND_1Y': get_bond_curve_data(cursor, 'L001618296', bd, ed),
        'URBAN_BOND_5Y': get_bond_curve_data(cursor, 'L003759264', bd, ed)
    }
    
    # 合并所有数据
    merged_data = usd_cny.copy()
    for name, df in data_dict.items():
        df = df.rename(columns={'value': name})
        merged_data = pd.merge(merged_data, df, on='date', how='outer')
    
    # 按日期排序
    merged_data = merged_data.sort_values('date')
    
    # 处理缺失值 - 使用前向填充和后向填充
    merged_data = merged_data.fillna(method='ffill').fillna(method='bfill')
    
    return merged_data

# 设置时间范围
bd = ANALYSIS_CONFIG['start_date']
ed = ANALYSIS_CONFIG['end_date']

# 获取数据
if db_conn is not None:
    print(f"正在获取数据，时间范围: {bd} 至 {ed}")
    data = get_all_data(db_conn, bd, ed)
    print(f"数据获取完成，共 {len(data)} 条记录")
else:
    # 尝试读取本地缓存数据
    cache_path = 'data/processed_data.parquet'
    if os.path.exists(cache_path):
        data = pd.read_parquet(cache_path)
        print(f"从本地缓存读取数据，共 {len(data)} 条记录")
    else:
        print("无可用数据源，请检查数据库连接或本地缓存")

---

## 3. 数据处理

数据清洗、质量检查和预处理

In [ ]:
# 数据质量检查
def check_data_quality(df):
    """数据质量检查"""
    print("=" * 60)
    print("数据质量检查报告")
    print("=" * 60)
    
    # 时间范围
    print(f"\n数据时间范围：")
    print(f"  开始日期：{df['date'].min()}")
    print(f"  结束日期：{df['date'].max()}")
    print(f"  数据点数量：{len(df)}")
    
    # 缺失值检查
    missing = df.isnull().sum()
    if missing.any():
        print(f"\n存在缺失值的列：")
        for col, count in missing[missing > 0].items():
            print(f"  {col}: {count} ({count/len(df)*100:.2f}%)")
    else:
        print(f"\n无缺失值")
    
    # 基本统计信息
    print(f"\n各列基本统计信息：")
    display(df.describe().round(4))
    
    return True

check_data_quality(data)

In [ ]:
# 查看数据前几行
print("数据预览：")
display(data.head(10))

In [ ]:
# 保存处理后的数据
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

data_path = os.path.join(output_dir, 'processed_data.parquet')
data.to_parquet(data_path, index=False)
print(f"数据已保存到: {data_path}")

---

## 4. 核心逻辑

相关性分析、滚动相关性计算、高相关性时期识别

In [ ]:
def calculate_rolling_correlation(data, window=60):
    """
    计算滚动相关性
    
    参数:
        data: 包含汇率和其他因子的DataFrame
        window: 滚动窗口大小（交易日）
    
    返回:
        滚动相关性DataFrame
    """
    correlations = {}
    target = 'USD_CNY'
    
    # 计算每个变量与汇率的滚动相关性
    for column in data.columns:
        if column != target and column != 'date':
            rolling_corr = data[target].rolling(window=window).corr(data[column])
            correlations[column] = rolling_corr
    
    # 将结果转换为DataFrame
    rolling_corr_df = pd.DataFrame(correlations, index=data.index)
    rolling_corr_df['date'] = data['date'].values
    
    return rolling_corr_df

def identify_high_correlation_periods(rolling_corr_df, threshold=0.5):
    """
    识别高相关性时期（只保留正相关）
    
    参数:
        rolling_corr_df: 滚动相关性DataFrame
        threshold: 相关性阈值
    
    返回:
        高相关性时期字典
    """
    high_corr_periods = {}
    
    for column in rolling_corr_df.columns:
        if column != 'date':
            # 获取相关系数大于阈值的时期（只保留正相关）
            high_corr = rolling_corr_df[['date', column]].copy()
            high_corr['is_high'] = (high_corr[column] >= threshold)
            
            # 识别连续的高相关期
            high_corr['period_group'] = (high_corr['is_high'] != high_corr['is_high'].shift()).cumsum()
            
            periods = []
            for group in high_corr[high_corr['is_high']]['period_group'].unique():
                period_data = high_corr[high_corr['period_group'] == group]
                if len(period_data) > 0:
                    start_date = period_data['date'].iloc[0]
                    end_date = period_data['date'].iloc[-1]
                    avg_corr = period_data[column].mean()
                    periods.append({
                        'start_date': start_date,
                        'end_date': end_date,
                        'avg_correlation': avg_corr
                    })
            
            high_corr_periods[column] = periods
    
    return high_corr_periods

print("核心分析函数定义完成！")

In [ ]:
# 执行相关性分析
rolling_window = ANALYSIS_CONFIG['rolling_window']
corr_threshold = ANALYSIS_CONFIG['correlation_threshold']

print(f"计算滚动相关性，窗口大小: {rolling_window}天")
rolling_corr_df = calculate_rolling_correlation(data, window=rolling_window)

print(f"识别高相关性时期，阈值: {corr_threshold}")
high_corr_periods = identify_high_correlation_periods(rolling_corr_df, threshold=corr_threshold)

# 计算整体相关性
overall_corr = data.drop('date', axis=1).corr()['USD_CNY'].drop('USD_CNY')
print("\n整体相关性（与USD_CNY）：")
display(overall_corr.to_frame().round(4))

In [ ]:
# 显示高相关性时期统计
print("\n各因子高相关性时期统计：")
print("=" * 60)
for factor, periods in high_corr_periods.items():
    print(f"\n{factor}: {len(periods)} 个高相关性时期")
    if len(periods) > 0:
        for i, p in enumerate(periods[:3], 1):  # 只显示前3个
            print(f"  {i}. {p['start_date'].strftime('%Y-%m-%d')} ~ {p['end_date'].strftime('%Y-%m-%d')}, 平均相关性: {p['avg_correlation']:.3f}")
        if len(periods) > 3:
            print(f"  ... 还有 {len(periods) - 3} 个时期")

---

## 5. 执行与测试

生成可视化报告并验证结果

In [ ]:
def create_correlation_report(data, rolling_corr_df, high_corr_periods, overall_corr):
    """创建相关性分析报告图表"""
    # 创建主图表
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('各因子与汇率的整体相关性', '各因子与汇率的动态相关性'),
        vertical_spacing=0.2,
        row_heights=[0.3, 0.7]
    )
    
    # 添加整体相关性柱状图
    fig.add_trace(
        go.Bar(
            x=overall_corr.index,
            y=overall_corr.values,
            name='整体相关性',
            text=overall_corr.values.round(3),
            textposition='auto',
        ),
        row=1, col=1
    )
    
    # 添加动态相关性曲线
    for column in rolling_corr_df.columns:
        if column != 'date':
            fig.add_trace(
                go.Scatter(
                    x=rolling_corr_df['date'],
                    y=rolling_corr_df[column],
                    name=column,
                    mode='lines',
                ),
                row=2, col=1
            )
    
    # 更新布局
    fig.update_layout(
        title='汇率因子相关性分析',
        showlegend=True,
        height=1000,
        template='plotly_white',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.05
        ),
        annotations=[
            dict(
                text="数据来源：弘则研究，WIND，同花顺",
                xref="paper",
                yref="paper",
                x=0.5,
                y=-0.1,
                showarrow=False,
                font=dict(size=10)
            )
        ]
    )
    
    # 更新x轴和y轴标签
    fig.update_xaxes(title_text="因子", row=1, col=1)
    fig.update_xaxes(title_text="日期", row=2, col=1)
    fig.update_yaxes(title_text="相关系数", row=1, col=1)
    fig.update_yaxes(title_text="相关系数", row=2, col=1)
    
    return fig

def create_factor_time_series(data, factor, high_corr_periods):
    """创建单个因子的时间序列对比图"""
    subfig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # 添加因子数据
    subfig.add_trace(
        go.Scatter(
            x=data['date'],
            y=data[factor],
            name=factor,
            line=dict(color='blue')
        ),
        secondary_y=False
    )
    
    # 添加汇率数据
    subfig.add_trace(
        go.Scatter(
            x=data['date'],
            y=data['USD_CNY'],
            name='USD_CNY',
            line=dict(color='red')
        ),
        secondary_y=True
    )
    
    # 添加高相关性时期的阴影
    if factor in high_corr_periods:
        for period in high_corr_periods[factor]:
            subfig.add_vrect(
                x0=period['start_date'],
                x1=period['end_date'],
                fillcolor="gray",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
    
    # 更新布局
    subfig.update_layout(
        title=f'{factor} vs USD_CNY (阴影区域表示高相关性时期)',
        xaxis_title="日期",
        height=400,
        showlegend=True
    )
    
    subfig.update_yaxes(title_text=factor, secondary_y=False)
    subfig.update_yaxes(title_text="USD_CNY", secondary_y=True)
    
    return subfig

print("可视化函数定义完成！")

In [ ]:
# 生成主报告图表
main_fig = create_correlation_report(data, rolling_corr_df, high_corr_periods, overall_corr)
main_fig.show()

In [ ]:
# 生成各因子时间序列图
time_series_figs = {}
for column in data.columns:
    if column not in ['date', 'USD_CNY']:
        fig = create_factor_time_series(data, column, high_corr_periods)
        time_series_figs[column] = fig
        print(f"生成 {column} 时间序列图")

print(f"\n共生成 {len(time_series_figs)} 个因子时间序列图")

In [ ]:
# 显示部分因子图表
for factor in list(time_series_figs.keys())[:3]:
    print(f"\n{factor} 与汇率对比：")
    time_series_figs[factor].show()

In [ ]:
def generate_html_report(main_fig, time_series_figs, high_corr_periods):
    """生成HTML报告"""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>汇率因子相关性分析报告</title>
        <meta charset="utf-8">
        <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
        <style>
            body { font-family: "Microsoft YaHei", Arial, sans-serif; margin: 40px; }
            .container { max-width: 1200px; margin: auto; }
            .section { margin-bottom: 40px; }
            .plot-container { width: 100%; height: 600px; margin: 20px 0; }
            table { border-collapse: collapse; width: 100%; margin: 20px 0; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>汇率因子相关性分析报告</h1>
            
            <div class="section">
                <h2>1. 整体相关性分析</h2>
                <div id="main-plot" class="plot-container"></div>
            </div>

            <div class="section">
                <h2>2. 各因子与汇率的时间序列分析</h2>
                <p>阴影区域表示相关系数大于0.5的正相关时期</p>
    """
    
    # 添加每个因子的时间序列图
    for i, (factor, fig) in enumerate(time_series_figs.items(), 1):
        html_content += f"""
                <div class="section">
                    <h3>{i}. {factor}</h3>
                    <div id="plot-{i}" class="plot-container"></div>
                    <h4>高相关性时期：</h4>
                    <table>
                        <tr>
                            <th>开始日期</th>
                            <th>结束日期</th>
                            <th>平均相关系数</th>
                        </tr>
        """
        
        if factor in high_corr_periods:
            for period in high_corr_periods[factor]:
                html_content += f"""
                        <tr>
                            <td>{period['start_date'].strftime('%Y-%m-%d')}</td>
                            <td>{period['end_date'].strftime('%Y-%m-%d')}</td>
                            <td>{period['avg_correlation']:.3f}</td>
                        </tr>
                """
        
        html_content += """
                    </table>
                </div>
        """
    
    html_content += """
            </div>
        </div>
    """
    
    # 添加图表数据
    html_content += f"<script>var mainPlot = {main_fig.to_json()};\n"
    for i, (factor, fig) in enumerate(time_series_figs.items(), 1):
        html_content += f"var plot{i} = {fig.to_json()};\n"
    
    # 添加绘图命令
    html_content += "Plotly.newPlot('main-plot', mainPlot.data, mainPlot.layout);\n"
    
    for i in range(1, len(time_series_figs) + 1):
        html_content += f"Plotly.newPlot('plot-{i}', plot{i}.data, plot{i}.layout);\n"
    
    html_content += """
        </script>
    </body>
    </html>
    """
    
    return html_content

# 生成并保存HTML报告
html_content = generate_html_report(main_fig, time_series_figs, high_corr_periods)
report_path = os.path.join(output_dir, 'correlation_analysis.html')

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\nHTML报告已生成: {report_path}")

---

## 6. 工具函数

可复用的工具函数

In [ ]:
def save_data_to_parquet(df, filepath):
    """保存数据到Parquet文件"""
    df.to_parquet(filepath, index=False)
    print(f"数据已保存: {filepath}")

def load_data_from_parquet(filepath):
    """从Parquet文件加载数据"""
    df = pd.read_parquet(filepath)
    print(f"数据已加载: {filepath}, 共 {len(df)} 条记录")
    return df

def get_latest_value(df, date_col, value_col):
    """获取最新值"""
    latest_date = df[date_col].max()
    latest = df[df[date_col] == latest_date][value_col].values[0]
    return latest_date, latest

def calculate_statistics(df, columns):
    """计算统计信息"""
    stats = df[columns].describe().T
    stats['missing'] = df[columns].isnull().sum()
    return stats

print("工具函数定义完成！")

In [ ]:
# 关闭数据库连接
if db_conn is not None:
    db_conn.close()
    print("数据库连接已关闭")

print("\n" + "="*60)
print("汇率因子检验分析完成！")
print("="*60)
print(f"\n输出文件：")
print(f"  - 数据文件: {data_path}")
print(f"  - 分析报告: {report_path}")